# Оптически достоверная синтетика мазка крови (малярия) + авторазметка
Идея: сначала строим карту **оптической плотности** (OD) сцены для каждого цветового канала
(клетки аддитивно поглощают свет, перекрытия получаются физически корректными, закон Бугера–Ламберта),
затем «снимаем» её виртуальным микроскопом: PSF/дефокус, поле освещения и виньетирование,
хроматическая аберрация, фотонный (Пуассон) + считывающий (Гаусс) шум, JPEG.
Разметка (bbox) считается из масок клеток, поэтому всегда точная.

Классы: `rbc`, `infected_rbc` (бокс хозяйской клетки), `wbc`, `platelet`.
Выход: YOLO-разметка, COCO json, data.yaml, meta.jsonl (seed каждой картинки для воспроизводимости).

In [ ]:
import os, json, math
from dataclasses import dataclass, field
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
CLASSES = ["rbc", "infected_rbc", "wbc", "platelet"]
COLORS = {0: (0, 114, 255), 1: (255, 0, 0), 2: (0, 170, 0), 3: (255, 160, 0)}


@dataclass
class Cfg:
    out_dir: str = "/content/synth_malaria"
    n_images: int = 1000
    split: tuple = (0.7, 0.2, 0.1)          # train / val / test
    W: int = 1024
    H: int = 1024
    seed: int = 42
    rbc_r: tuple = (24.0, 32.0)             # радиус эритроцита, px (меняется от кадра к кадру)
    n_rbc: tuple = (60, 120)                # целевое число эритроцитов на кадр
    n_wbc: tuple = (0, 2)
    n_plt: tuple = (0, 6)
    p_negative: float = 0.15                # доля кадров вообще без паразитов
    p_inf: tuple = (0.03, 0.30)             # доля заражённых эритроцитов в положительном кадре
    stage_p: tuple = (0.55, 0.30, 0.15)     # кольцо / трофозоит / шизонт
    min_vis: float = 0.5                    # минимальная видимая доля клетки (для обрезанных краем кадра)
    optics: dict = field(default_factory=lambda: dict(
        psf=True, vignette=True, chroma=True, noise=True, jpeg=True))


cfg = Cfg()

## 1. Сцена: клетки в пространстве оптической плотности

In [ ]:
# «Оптическая плотность» на единицу толщины по каналам R,G,B (Гимза-подобная окраска)
BASE_OD = dict(
    rbc=(0.36, 0.46, 0.40),    # гемоглобин: серо-розовый
    cyto=(0.60, 0.42, 0.12),   # цитоплазма паразита: голубая (сильно гасит красный)
    chrom=(0.55, 0.95, 0.45),  # хроматин паразита: пурпурно-красный
    pig=(0.45, 0.65, 1.00),    # гемозоин: коричневатый
    wcyto=(0.10, 0.16, 0.10),  # цитоплазма лейкоцита
    nuc=(0.75, 1.00, 0.40),    # ядро лейкоцита: фиолетовое
    plt=(0.45, 0.65, 0.35),    # тромбоциты
)


def sample_stain(rng):
    """Межпрепаратная вариативность окраски/толщины мазка."""
    g = rng.uniform(0.8, 1.25)
    return {k: (np.array(v) * g * rng.uniform(0.9, 1.1, 3)).astype(np.float32)
            for k, v in BASE_OD.items()}


def smooth_noise(H, W, scale, rng):
    h, w = max(2, int(H / scale) + 2), max(2, int(W / scale) + 2)
    n = rng.random((h, w)).astype(np.float32)
    n = cv2.resize(n, (W, H), interpolation=cv2.INTER_CUBIC)
    n -= n.min()
    return n / (n.max() + 1e-6)


def patch(cx, cy, R, W, H):
    """Окно вокруг (cx,cy) радиуса R, обрезанное по кадру; возвращает срезы и координаты относительно центра."""
    x0, x1 = max(int(math.floor(cx - R)), 0), min(int(math.ceil(cx + R)) + 1, W)
    y0, y1 = max(int(math.floor(cy - R)), 0), min(int(math.ceil(cy + R)) + 1, H)
    if x1 <= x0 or y1 <= y0:
        return None
    xs = np.arange(x0, x1, dtype=np.float32) - cx
    ys = np.arange(y0, y1, dtype=np.float32) - cy
    dx, dy = np.meshgrid(xs, ys)
    return (y0, y1, x0, x1), dx, dy


def blob(dx, dy, mx, my, s):
    return np.exp(-(((dx - mx) ** 2 + (dy - my) ** 2) / (2 * s * s)))


def bbox_from_mask(mask, x0, y0, thr=0.5):
    ys, xs = np.nonzero(mask > thr)
    if len(xs) == 0:
        return None
    return (int(x0 + xs.min()), int(y0 + ys.min()), int(x0 + xs.max() + 1), int(y0 + ys.max() + 1))


def add_rbc(od, cx, cy, r, rng, V):
    """Эритроцит: слегка эллиптический, неровный край, центральное просветление (бикогнутость)."""
    H, W = od.shape[:2]
    a, b = r, r * rng.uniform(0.85, 1.0)
    th = rng.uniform(0, math.pi)
    p = patch(cx, cy, a * 1.3, W, H)
    if p is None:
        return None
    (y0, y1, x0, x1), dx, dy = p
    c, s = math.cos(th), math.sin(th)
    u, v = (dx * c + dy * s) / a, (-dx * s + dy * c) / b
    rho = np.sqrt(u * u + v * v) + 1e-6
    phi = np.arctan2(v, u)
    edge = np.ones_like(rho)
    for k in (2, 3, 4):
        edge += rng.uniform(0, 0.035 / (k - 1)) * np.cos(k * phi + rng.uniform(0, 2 * math.pi))
    rn = rho / edge
    mask = np.clip((1 - rn) * b + 0.5, 0, 1)                      # антиалиасинг края
    pallor, pw = rng.uniform(0.1, 0.55), rng.uniform(0.30, 0.45)  # глубина/ширина просветления
    t = mask * (1 - pallor * np.exp(-(rn / pw) ** 2)) * (1 + 0.12 * rn ** 4)  # толщина с утолщённым ободком
    od[y0:y1, x0:x1] += t[..., None] * V["rbc"]
    box = bbox_from_mask(mask, x0, y0)
    if box is None:
        return None
    return box, float(mask.sum() / (math.pi * a * b))


def add_parasite(od, cx, cy, r, rng, stage, V):
    """Plasmodium внутри эритроцита. stage: 0 кольцо, 1 трофозоит, 2 шизонт."""
    H, W = od.shape[:2]
    p = patch(cx, cy, r, W, H)
    if p is None:
        return
    (y0, y1, x0, x1), dx, dy = p
    inside = np.clip(r * 0.92 - np.hypot(dx, dy), 0, 1)
    cyto, chrom, pig = np.zeros_like(dx), np.zeros_like(dx), np.zeros_like(dx)
    ang, off = rng.uniform(0, 2 * math.pi), rng.uniform(0.0, 0.45) * r
    px, py = off * math.cos(ang), off * math.sin(ang)
    n_par = 2 if rng.random() < 0.08 else 1                       # множественное заражение
    for _ in range(n_par):
        if _ == 1:
            px, py = -px * 0.8, -py * 0.8
        if stage == 0:    # кольцо с хроматиновой точкой
            rr, w = rng.uniform(0.18, 0.30) * r, max(1.2, 0.07 * r)
            cyto += np.exp(-((np.hypot(dx - px, dy - py) - rr) / w) ** 2)
            for _d in range(1 + int(rng.random() < 0.25)):
                a2 = rng.uniform(0, 2 * math.pi)
                chrom += blob(dx, dy, px + rr * math.cos(a2), py + rr * math.sin(a2), max(1.2, 0.075 * r))
        elif stage == 1:  # амёбоидный трофозоит
            for _b in range(rng.integers(2, 4)):
                cyto += blob(dx, dy, px + rng.normal(0, 0.12 * r), py + rng.normal(0, 0.12 * r),
                             rng.uniform(0.16, 0.26) * r)
            cyto -= 0.5 * blob(dx, dy, px, py, 0.08 * r)           # вакуоль
            chrom += blob(dx, dy, px + rng.normal(0, 0.12 * r), py + rng.normal(0, 0.12 * r), max(1.3, 0.08 * r))
            for _p in range(rng.integers(0, 4)):
                pig += blob(dx, dy, px + rng.normal(0, 0.15 * r), py + rng.normal(0, 0.15 * r), max(0.8, 0.03 * r))
        else:             # шизонт: много мерозоитов + пигмент
            cyto += 0.8 * blob(dx, dy, px * 0.5, py * 0.5, 0.38 * r)
            for _m in range(rng.integers(6, 13)):
                chrom += blob(dx, dy, px * 0.5 + rng.normal(0, 0.2 * r), py * 0.5 + rng.normal(0, 0.2 * r),
                              max(1.0, 0.06 * r))
            pig += 1.5 * blob(dx, dy, px * 0.5 + rng.normal(0, 0.1 * r), py * 0.5 + rng.normal(0, 0.1 * r),
                              max(1.2, 0.07 * r))
    cyto, chrom, pig = np.clip(cyto, 0, 1.3), np.clip(chrom, 0, 1.3), np.clip(pig, 0, 1.5)
    add = (cyto[..., None] * V["cyto"] * 0.9 + chrom[..., None] * V["chrom"] * 1.3
           + pig[..., None] * V["pig"]) * inside[..., None]
    od[y0:y1, x0:x1] += add.astype(np.float32)


def add_wbc(od, cx, cy, r, rng, V):
    """Лейкоцит: крупная клетка с многодольчатым ядром и зернистой текстурой хроматина."""
    H, W = od.shape[:2]
    p = patch(cx, cy, r * 1.3, W, H)
    if p is None:
        return None
    (y0, y1, x0, x1), dx, dy = p
    phi = np.arctan2(dy, dx)
    edge = np.ones_like(dx)
    for k in (2, 3, 5):
        edge += rng.uniform(0, 0.05) * np.cos(k * phi + rng.uniform(0, 2 * math.pi))
    cmask = np.clip(r * edge - np.hypot(dx, dy) + 0.5, 0, 1)
    nuc = np.zeros_like(dx)
    n_lobes = int(rng.integers(1, 5))
    lr = r * (rng.uniform(0.38, 0.5) if n_lobes == 1 else rng.uniform(0.26, 0.36))
    pos = np.array([rng.normal(0, 0.15 * r), rng.normal(0, 0.15 * r)])
    d = rng.uniform(0, 2 * math.pi)
    for _ in range(n_lobes):
        nuc = np.maximum(nuc, np.clip(lr - np.hypot(dx - pos[0], dy - pos[1]) + 0.5, 0, 1))
        d += rng.normal(0, 0.7)
        pos = pos + 0.95 * lr * np.array([math.cos(d), math.sin(d)])
    tex = cv2.resize(rng.random((7, 7)).astype(np.float32), (dx.shape[1], dx.shape[0]), interpolation=cv2.INTER_CUBIC)
    nuc = nuc * (0.75 + 0.5 * tex) * cmask
    od[y0:y1, x0:x1] += cmask[..., None] * V["wcyto"] + nuc[..., None] * V["nuc"] * 1.1
    box = bbox_from_mask(cmask, x0, y0)
    if box is None:
        return None
    return box, float(cmask.sum() / (math.pi * r * r))


def add_platelet(od, cx, cy, r, rng, V):
    H, W = od.shape[:2]
    p = patch(cx, cy, r * 1.4, W, H)
    if p is None:
        return None
    (y0, y1, x0, x1), dx, dy = p
    phi = np.arctan2(dy, dx)
    edge = 1 + rng.uniform(0, 0.15) * np.cos(2 * phi + rng.uniform(0, 6.28)) + rng.uniform(0, 0.1) * np.cos(3 * phi)
    mask = np.clip(r * edge - np.hypot(dx, dy) + 0.5, 0, 1)
    tex = cv2.resize(rng.random((5, 5)).astype(np.float32), (dx.shape[1], dx.shape[0]), interpolation=cv2.INTER_CUBIC)
    od[y0:y1, x0:x1] += (mask * (0.6 + 0.8 * tex))[..., None] * V["plt"]
    box = bbox_from_mask(mask, x0, y0)
    if box is None:
        return None
    return box, float(mask.sum() / (math.pi * r * r))


def add_dust(od, cx, cy, s, rng):
    """Артефакты (пыль/преципитат окраски) — НЕ размечаются, служат hard negatives."""
    H, W = od.shape[:2]
    p = patch(cx, cy, 4 * s, W, H)
    if p is None:
        return
    (y0, y1, x0, x1), dx, dy = p
    col = rng.choice([0, 1], p=[0.6, 0.4])
    vec = np.array([0.5, 0.5, 0.5] if col == 0 else [0.5, 0.9, 0.5], np.float32) * rng.uniform(0.3, 1.0)
    od[y0:y1, x0:x1] += blob(dx, dy, 0, 0, s)[..., None] * vec


class Placer:
    def __init__(self):
        self.P = np.zeros((1000, 3), np.float32)
        self.n = 0

    def free(self, x, y, r, f):
        if self.n == 0:
            return True
        d = np.hypot(self.P[:self.n, 0] - x, self.P[:self.n, 1] - y)
        return bool(np.all(d >= f * (self.P[:self.n, 2] + r)))

    def add(self, x, y, r):
        self.P[self.n] = (x, y, r)
        self.n += 1


def make_scene(cfg, rng):
    """Возвращает карту оптической плотности (H,W,3) и список боксов (cls,x1,y1,x2,y2)."""
    W, H = cfg.W, cfg.H
    od = np.zeros((H, W, 3), np.float32)
    V = sample_stain(rng)
    boxes, pl = [], Placer()
    r0 = rng.uniform(*cfg.rbc_r) * (W / 1024)       # «увеличение» кадра

    # --- лейкоциты (первыми, чтобы эритроциты их огибали)
    for _ in range(int(rng.integers(cfg.n_wbc[0], cfg.n_wbc[1] + 1))):
        r = r0 * rng.uniform(1.7, 2.2)
        for _t in range(30):
            x, y = rng.uniform(0, W), rng.uniform(0, H)
            if pl.free(x, y, r, 0.9):
                break
        else:
            continue
        pl.add(x, y, r)
        res = add_wbc(od, x, y, r, rng, V)
        if res and res[1] >= cfg.min_vis:
            boxes.append((2, *res[0]))

    # --- эритроциты: неоднородная плотность, допускаются частичные перекрытия
    dens = smooth_noise(H, W, rng.uniform(250, 500), rng)
    target = int(rng.integers(cfg.n_rbc[0], cfg.n_rbc[1] + 1))
    p_inf = 0.0 if rng.random() < cfg.p_negative else rng.uniform(*cfg.p_inf)
    rbcs, cnt = [], 0
    for _ in range(target * 60):
        if cnt >= target:
            break
        r = r0 * rng.uniform(0.9, 1.1)
        x, y = rng.uniform(-0.3 * r, W + 0.3 * r), rng.uniform(-0.3 * r, H + 0.3 * r)
        if rng.random() > 0.15 + 0.85 * dens[int(np.clip(y, 0, H - 1)), int(np.clip(x, 0, W - 1))] ** 1.2:
            continue
        if not pl.free(x, y, r, float(rng.choice([0.8, 0.9, 1.0, 1.0]))):
            continue
        pl.add(x, y, r)
        cnt += 1
        res = add_rbc(od, x, y, r, rng, V)
        if res is None:
            continue
        box, vis = res
        rbcs.append((x, y, r))
        if vis < cfg.min_vis:
            continue
        infected = rng.random() < p_inf
        if infected:
            add_parasite(od, x, y, r, rng, int(rng.choice(3, p=cfg.stage_p)), V)
        boxes.append((1 if infected else 0, *box))

    # --- тромбоциты: рядом с эритроцитами
    for _ in range(int(rng.integers(cfg.n_plt[0], cfg.n_plt[1] + 1))):
        if not rbcs:
            break
        bx, by, br = rbcs[int(rng.integers(len(rbcs)))]
        a, rp = rng.uniform(0, 2 * math.pi), r0 * rng.uniform(0.15, 0.28)
        x, y = bx + (br + 0.3 * rp) * math.cos(a), by + (br + 0.3 * rp) * math.sin(a)
        res = add_platelet(od, x, y, rp, rng, V)
        if res and res[1] >= cfg.min_vis:
            boxes.append((3, *res[0]))

    # --- артефакты без разметки
    for _ in range(int(rng.integers(0, 20))):
        add_dust(od, rng.uniform(0, W), rng.uniform(0, H), rng.uniform(1.0, 3.5) * (W / 1024), rng)

    boxes = [b for b in boxes if (b[3] - b[1]) >= 4 and (b[4] - b[2]) >= 4]
    return od, boxes

## 2. Виртуальный микроскоп (оптика + сенсор)
Каждый эффект включается флагом в `cfg.optics` — удобно для абляции в статье.

In [ ]:
def render(od, cfg, opt, seed):
    H, W = od.shape[:2]
    sc = W / 1024
    # у каждого эффекта свой независимый генератор: параметры не «плывут» при включении/выключении других эффектов
    R = {k: np.random.default_rng([seed, i]) for i, k in enumerate(("base", "psf", "chroma", "vignette", "noise", "jpeg"))}
    rng = R["base"]
    bg = np.clip(np.array([0.80, 0.80, 0.82]) * rng.uniform(0.95, 1.05, 3), 0, 0.97).astype(np.float32)
    img = np.exp(-od) * bg                                      # закон Бугера–Ламберта
    ch = [np.ascontiguousarray(img[..., c]) for c in range(3)]

    rng = R["psf"]
    if opt.get("psf", True):    # дефокус (гауссов PSF) + кривизна поля: часть кадра размыта сильнее; B размыт сильнее R
        s = rng.uniform(0.8, 2.0) * sc
        m = smooth_noise(H, W, rng.uniform(300, 600), rng) * rng.uniform(0.0, 0.7)
        for c, k in enumerate((0.95, 1.0, 1.12)):
            sharp = cv2.GaussianBlur(ch[c], (0, 0), s * k)
            soft = cv2.GaussianBlur(ch[c], (0, 0), s * k * 2.2)
            ch[c] = sharp * (1 - m) + soft * m

    rng = R["chroma"]
    if opt.get("chroma", True):  # поперечная хроматическая аберрация: R и B масштабируются/сдвигаются в разные стороны
        eps = rng.uniform(0.0003, 0.0015)
        t = rng.uniform(-0.6, 0.6, 2) * sc
        for c, sg in ((0, 1), (2, -1)):
            s_ = 1 + sg * eps
            M = np.float32([[s_, 0, (1 - s_) * W / 2 + sg * t[0]], [0, s_, (1 - s_) * H / 2 + sg * t[1]]])
            ch[c] = cv2.warpAffine(ch[c], M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

    img = np.stack(ch, -1)

    rng = R["vignette"]
    if opt.get("vignette", True):  # неоднородное освещение: виньетирование + градиент + низкочастотная неравномерность
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        cx, cy = W * rng.uniform(0.4, 0.6), H * rng.uniform(0.4, 0.6)
        r2 = ((xx - cx) ** 2 + (yy - cy) ** 2) / ((W / 2) ** 2 + (H / 2) ** 2)
        illum = (1 - rng.uniform(0.05, 0.30) * r2 + rng.uniform(-0.06, 0.06) * (xx / W - 0.5)
                 + rng.uniform(-0.06, 0.06) * (yy / H - 0.5) + 0.04 * (smooth_noise(H, W, 400, rng) - 0.5))
        img *= illum[..., None]

    img = np.clip(img, 0, 1)
    rng = R["noise"]
    if opt.get("noise", True):     # фотонный шум Пуассона + гауссов шум считывания, затем коррелирование (демозаика)
        N = 10 ** rng.uniform(2.6, 3.6)
        img = rng.poisson(img * N).astype(np.float32) / N
        img += rng.normal(0, rng.uniform(0.003, 0.012), img.shape).astype(np.float32)
        img = cv2.GaussianBlur(img, (0, 0), 0.6)

    rng = R["base"]
    gain, gamma = rng.uniform(0.97, 1.03, 3), rng.uniform(0.9, 1.1)   # баланс белого / гамма камеры
    img = np.clip(img * gain, 0, 1) ** gamma
    u8 = (img * 255 + 0.5).astype(np.uint8)

    if opt.get("jpeg", True):
        q = int(R["jpeg"].integers(55, 95))
        _, buf = cv2.imencode(".jpg", np.ascontiguousarray(u8[..., ::-1]), [cv2.IMWRITE_JPEG_QUALITY, q])
        u8 = cv2.imdecode(buf, cv2.IMREAD_COLOR)[..., ::-1]
    return np.ascontiguousarray(u8)  # RGB

## 3. Генерация датасета (YOLO + COCO)

In [ ]:
def build_dataset(cfg):
    out = Path(cfg.out_dir)
    n = cfg.n_images
    perm = np.random.default_rng(cfg.seed).permutation(n)
    n_tr, n_va = int(n * cfg.split[0]), int(n * cfg.split[1])
    split_of = {}
    for k, i in enumerate(perm):
        split_of[int(i)] = "train" if k < n_tr else ("val" if k < n_tr + n_va else "test")
    for sp in ("train", "val", "test"):
        (out / "images" / sp).mkdir(parents=True, exist_ok=True)
        (out / "labels" / sp).mkdir(parents=True, exist_ok=True)

    cats = [{"id": i + 1, "name": c} for i, c in enumerate(CLASSES)]
    coco = {sp: {"images": [], "annotations": [], "categories": cats} for sp in ("train", "val", "test")}
    ann_id = 1
    meta = open(out / "meta.jsonl", "w")
    stats = np.zeros(len(CLASSES), int)

    for i in tqdm(range(n)):
        seed = cfg.seed * 1_000_000 + i
        sp = split_of[i]
        rng = np.random.default_rng(seed)
        od, boxes = make_scene(cfg, rng)
        img = render(od, cfg, cfg.optics, seed + 7)

        name = f"synth_{i:06d}"
        cv2.imwrite(str(out / "images" / sp / f"{name}.jpg"), np.ascontiguousarray(img[..., ::-1]),
                    [cv2.IMWRITE_JPEG_QUALITY, 98])
        with open(out / "labels" / sp / f"{name}.txt", "w") as f:
            for c, x1, y1, x2, y2 in boxes:
                f.write(f"{c} {(x1 + x2) / 2 / cfg.W:.6f} {(y1 + y2) / 2 / cfg.H:.6f} "
                        f"{(x2 - x1) / cfg.W:.6f} {(y2 - y1) / cfg.H:.6f}\n")
        coco[sp]["images"].append({"id": i, "file_name": f"{name}.jpg", "width": cfg.W, "height": cfg.H})
        for c, x1, y1, x2, y2 in boxes:
            coco[sp]["annotations"].append({"id": ann_id, "image_id": i, "category_id": c + 1,
                                            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1),
                                            "iscrowd": 0})
            ann_id += 1
            stats[c] += 1
        meta.write(json.dumps({"name": name, "split": sp, "seed": seed, "n_boxes": len(boxes)}) + "\n")
    meta.close()

    for sp in coco:
        with open(out / f"coco_{sp}.json", "w") as f:
            json.dump(coco[sp], f)
    with open(out / "data.yaml", "w") as f:
        f.write(f"path: {out}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: {len(CLASSES)}\n"
                f"names: {CLASSES}\n")
    print("Боксов по классам:", dict(zip(CLASSES, stats.tolist())))
    print("Готово ->", out)

In [ ]:
build_dataset(cfg)

## 4. Визуальная проверка разметки

In [ ]:
def show_samples(cfg, split="train", n=6, seed=0):
    out = Path(cfg.out_dir)
    files = sorted((out / "images" / split).glob("*.jpg"))
    pick = np.random.default_rng(seed).choice(len(files), min(n, len(files)), replace=False)
    cols = 3
    fig, axs = plt.subplots(math.ceil(len(pick) / cols), cols, figsize=(6 * cols, 6 * math.ceil(len(pick) / cols)))
    for ax, k in zip(np.atleast_1d(axs).ravel(), pick):
        f = files[k]
        im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        for line in open(out / "labels" / split / (f.stem + ".txt")):
            c, cx, cy, bw, bh = line.split()
            c, cx, cy, bw, bh = int(c), float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
            cv2.rectangle(im, (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2)),
                          COLORS[c], 2)
        ax.imshow(im)
        ax.set_title(f.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_samples(cfg, "train", 6)

## 5. Абляция оптики для статьи
Одна и та же сцена, эффекты включаются кумулятивно — готовая иллюстрация «вклад каждого оптического эффекта».

In [ ]:
def ablation_figure(cfg, seed=123, crop=(150, 150, 550, 550)):
    rng = np.random.default_rng(seed)
    od, boxes = make_scene(cfg, rng)
    steps = [("идеальный рендер", {}), ("+ PSF/дефокус", dict(psf=True)),
             ("+ хром. аберрация", dict(psf=True, chroma=True)),
             ("+ виньетирование", dict(psf=True, chroma=True, vignette=True)),
             ("+ шум сенсора", dict(psf=True, chroma=True, vignette=True, noise=True)),
             ("+ JPEG (полная)", dict(psf=True, chroma=True, vignette=True, noise=True, jpeg=True))]
    x1, y1, x2, y2 = crop
    fig, axs = plt.subplots(2, 3, figsize=(15, 10))
    for ax, (title, o) in zip(axs.ravel(), steps):
        full = {k: o.get(k, False) for k in ("psf", "chroma", "vignette", "noise", "jpeg")}
        im = render(od, cfg, full, seed + 7)
        ax.imshow(im[y1:y2, x1:x2])
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


ablation_figure(cfg)

## 6. (Опционально) обучение лёгкой модели
Раскомментируйте. Ключевой эксперимент для статьи: **train на синтетике → test на реальных** снимках
(и сравнение с вариантами без оптических эффектов из п.5, а также с обучением на малой реальной выборке).

In [ ]:
# !pip -q install ultralytics
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")
# model.train(data=f"{cfg.out_dir}/data.yaml", imgsz=640, epochs=50, batch=16)
# model.val(data=f"{cfg.out_dir}/data.yaml", split="test")